# Q-learning (model-free)_Artificial Intelligence (CS221)_**No map of the world? Learn the value of actions by trying them and seeing what happens.**Value iteration needs the transition probabilities $T$. Often we do not have them.     Q-learning learns Q-values directly from experience. No model needed.---This notebook builds Q-learning from scratch, one piece at a time. The agent never sees the transition probabilities — it only acts, observes a reward, and nudges its Q-estimates toward what it saw. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPyWe'll teach an agent to walk a 4-state chain to the goal using **Q-learning**. The agent has no map of the world: it only learns by acting and observing rewards. We build it in three steps: (1) define the environment, (2) run the learning loop, (3) read the learned policy.

### Step 1 — Define the environment and the Q-tableThe world is a line of 4 states, `0` through `3`, where state `3` is the goal. The agent can go `left` (action 0) or `right` (action 1); moving off either end just keeps it in place. Landing on the goal pays `+10`, every other move costs `-1`, so the agent is pushed to reach the goal quickly. `Q` starts at all zeros — the agent knows nothing yet.

In [ ]:
rng = np.random.default_rng(0)   # seeded so results are reproducible

# Chain: states 0..3, state 3 is the goal. Action 0 = left, 1 = right.
N, A = 4, 2

def step(s, a):
    if a == 0:
        s2 = max(0, s - 1)            # move left, clamped at 0
    else:
        s2 = min(N - 1, s + 1)        # move right, clamped at the goal
    reward = 10.0 if s2 == N - 1 else -1.0
    return s2, reward

Q = np.zeros((N, A))                  # one Q-value per (state, action); start blank
alpha, gamma, eps = 0.5, 0.9, 0.2     # learning rate, discount, exploration rate

### Step 2 — Run the Q-learning loopEach episode the agent starts at state `0` and acts for up to 20 steps. It picks actions with an **epsilon-greedy** rule: with probability `eps` it explores a random action, otherwise it greedily takes the action with the highest current Q-value. After each move it applies the **Q-learning update**, nudging `Q[s, a]` toward the observed reward plus the discounted best value of the next state. `gamma` discounts future reward; `alpha` controls how big each nudge is.

In [ ]:
for ep in range(400):
    s = 0
    for _ in range(20):
        # Epsilon-greedy: explore at random, else take the current best action.
        if rng.random() < eps:
            a = rng.integers(A)
        else:
            a = int(Q[s].argmax())

        s2, r = step(s, a)

        # Q-learning update toward r + gamma * best next Q.
        best_next = Q[s2].max()
        td_target = r + gamma * best_next
        Q[s, a] += alpha * (td_target - Q[s, a])

        s = s2
        if s == N - 1:
            break

### Step 3 — Read the learned Q-table and policyThe greedy policy at each state is just the action with the larger Q-value. If learning worked, every state should prefer `right` (action 1), since that is the path toward the goal.

In [ ]:
print("learned Q-table:\n", np.round(Q, 2))

greedy_policy = Q.argmax(axis=1)
print("greedy policy (0=left,1=right):", greedy_policy)

## Visualize the data & results_Can a warehouse robot learn its way to the goal shelf by trial and error, never told the floor layout?_

### Step 1 — Lay out the warehouse gridThe floor is a 3×4 grid with a `WALL` the robot can't enter, a `GOAL` shelf paying `+1`, and a `HAZARD` paying `-1`. Both the goal and hazard end the episode. Every ordinary move costs a small `-0.04` to encourage short paths. The robot starts each episode at the charging dock and knows none of this — it learns purely from rewards.

In [ ]:
ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1, 1), (0, 3), (1, 3)
gamma = 0.95
acts = [(-1, 0), (1, 0), (0, -1), (0, 1)]      # up, down, left, right

def ok(s):
    return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s != WALL

states = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) != WALL]

def step(s, ai):
    d = acts[ai]
    ns = (s[0] + d[0], s[1] + d[1])
    if not ok(ns):
        ns = s                                  # bumping a wall/edge: stay put
    if ns == GOAL:
        return ns, 1.0, True
    if ns == HAZARD:
        return ns, -1.0, True
    return ns, -0.04, False

### Step 2 — Train the robot with Q-learningSame algorithm as the chain, now over a dictionary of `(state, action)` Q-values across 3000 episodes. The robot starts at the dock `(2, 0)`, acts epsilon-greedily, and applies the Q-learning update each step. When an episode ends (goal or hazard), there is no next state, so the future value is `0`.

In [ ]:
rng = np.random.default_rng(1)
Q = {(s, a): 0.0 for s in states for a in range(4)}
alpha, eps = 0.5, 0.2

for ep in range(3000):
    s = (2, 0)                          # charging dock
    for t in range(50):
        greedy = int(np.argmax([Q[(s, i)] for i in range(4)]))
        if rng.random() < eps:
            a = rng.integers(4)
        else:
            a = greedy

        ns, r, done = step(s, a)

        nxt = 0 if done else max(Q[(ns, i)] for i in range(4))
        Q[(s, a)] += alpha * (r + gamma * nxt - Q[(s, a)])

        s = ns
        if done:
            break

### Step 3 — Visualise the learned value of each cellFor every cell we take the best Q-value over its four actions — that is how good the robot thinks standing there is. We overlay the fixed goal and hazard values and draw it as a heatmap. Cells nearer the goal should glow brighter, tracing the path the robot learned.

In [ ]:
grid = np.full((ROWS, COLS), np.nan)
for s in states:
    grid[s] = max(Q[(s, i)] for i in range(4))
grid[GOAL] = 1.0
grid[HAZARD] = -1.0

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, cmap="viridis")

for r in range(ROWS):
    for c in range(COLS):
        if (r, c) != WALL:
            ax.text(c, r, round(grid[r, c], 2), ha="center", va="center", color="white")

ax.set_title("warehouse floor: best learned Q per cell")
fig.colorbar(im, ax=ax)
plt.show()